In [1]:
!git clone https://github.com/nicher92/swedish_correctness_classifier.git
%cd swedish_correctness_classifier
!ls

Cloning into 'swedish_correctness_classifier'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 36 (delta 15), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 1.70 MiB | 9.29 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/swedish_correctness_classifier
data  evaluate.py  index.html  predict.py  README.md  train.py


In [2]:
!python train.py

config.json: 100% 1.22k/1.22k [00:00<00:00, 2.67MB/s]
tokenizer_config.json: 100% 1.56k/1.56k [00:00<00:00, 3.62MB/s]
tokenizer.json: 100% 3.61M/3.61M [00:00<00:00, 72.5MB/s]
special_tokens_map.json: 100% 1.02k/1.02k [00:00<00:00, 2.82MB/s]

model.safetensors: downloading bytes:  15% 232M/1.58G [00:02<00:06, 216MB/s, 19.4MB/s  ]
model.safetensors: downloading bytes:  16% 260M/1.58G [00:02<00:11, 115MB/s, 22.3MB/s  ]
model.safetensors: downloading bytes:  35% 557M/1.58G [00:03<00:03, 316MB/s, 44.2MB/s  ]
model.safetensors: downloading bytes:  80% 1.26G/1.58G [00:06<00:01, 287MB/s, 94.3MB/s  ]
model.safetensors: downloading bytes:  94% 1.49G/1.58G [00:09<00:00, 161MB/s, 89.2MB/s  ]
model.safetensors: reconstructing file:  85% 1.34G/1.58G [00:16<00:02, 84.1MB/s, 36.4MB/s  ]
model.safetensors: downloading bytes: 100% 1.50G/1.50G [00:17<00:00, 87.0MB/s, 90.4MB/s  ]
model.safetensors: reconstructing file: 100% 1.58G/1.58G [00:17<00:00, 92.0MB/s, 94.3MB/s  ]
Loading weights: 100% 170/170 [00:

In [3]:
!python evaluate.py

Loading weights: 100% 170/170 [00:00<00:00, 14710.18it/s]
[transformers] ModernBertModel LOAD REPORT from: AI-Sweden-Models/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

120 test examples, timing covers the whole predict.py run including model loading

| Spearman | ROC-AUC | Time (s) | Samples/min |
|---|---|---|---|
| 0.528 | 0.783 | 70.2 | 103 |


In [4]:
import json

with open("data/train.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

pair = next(
    row["group"] for row in rows
    if row["label"] == "poor"
    and any(r["group"] == row["group"] and r["label"] == "good" for r in rows)
)

for row in rows:
    if row["group"] == pair:
        print(row["label"], row["score"], row["text"])

poor 0.45 De sålde deras hus förra året.
good 0.95 De sålde sitt hus förra året.


In [5]:
import json
from itertools import product
from pathlib import Path

original = Path("data/train.jsonl")
rows = [json.loads(line) for line in original.read_text(encoding="utf-8").splitlines()]
seen = {row["text"].strip().casefold() for row in rows}

nouns = [
    ("en", "bok"), ("en", "cykel"), ("en", "stol"),
    ("en", "lampa"), ("en", "väska"),
    ("ett", "äpple"), ("ett", "bord"), ("ett", "paket"),
    ("ett", "paraply"), ("ett", "fönster"),
]
starts = ["Jag köpte", "Hon valde", "Vi såg", "Ali hittade", "Sara visade"]
swap = {"en": "ett", "ett": "en"}
extra = []

for start, (article, noun) in product(starts, nouns):
    good = f"{start} {article} {noun}."
    poor = f"{start} {swap[article]} {noun}."
    keys = {good.casefold(), poor.casefold()}
    if keys & seen:
        continue

    group = f"article-{len(extra) // 2 + 1}"
    extra.extend([
        {"id": f"{group}-good", "group": group, "text": good,
         "score": 0.95, "label": "good", "error": ""},
        {"id": f"{group}-poor", "group": group, "text": poor,
         "score": 0.45, "label": "poor", "error": "wrong en/ett article"},
    ])
    seen.update(keys)

out = Path("data/train_augmented.jsonl")
with out.open("w", encoding="utf-8") as f:
    for row in rows + extra:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Original: {len(rows)} | Added: {len(extra)} | Total: {len(rows) + len(extra)}")
print(f"Saved: {out}")

Original: 478 | Added: 100 | Total: 578
Saved: data/train_augmented.jsonl


In [6]:
from pathlib import Path
import shutil
import subprocess
import sys

experiment = Path("experiment_augmented")
(experiment / "data").mkdir(parents=True, exist_ok=True)

for script in ("train.py", "predict.py"):
    shutil.copy2(script, experiment / script)

shutil.copy2(
    "data/train_augmented.jsonl",
    experiment / "data" / "train.jsonl",
)

subprocess.run([sys.executable, "train.py"], cwd=experiment, check=True)

CompletedProcess(args=['/usr/bin/python3', 'train.py'], returncode=0)

In [7]:
!python evaluate.py --repo experiment_augmented --output scores_augmented.jsonl

Loading weights: 100% 170/170 [00:00<00:00, 3588.97it/s]
[transformers] ModernBertModel LOAD REPORT from: AI-Sweden-Models/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

120 test examples, timing covers the whole predict.py run including model loading

| Spearman | ROC-AUC | Time (s) | Samples/min |
|---|---|---|---|
| 0.552 | 0.788 | 66.6 | 108 |


In [8]:
import json

def read_by_id(path):
    with open(path, encoding="utf-8") as f:
        return {row["id"]: row for row in map(json.loads, f)}

texts = read_by_id("data/test.jsonl")
labels = read_by_id("data/test_labels.jsonl")
predictions = read_by_id("scores_augmented.jsonl")

for label, direction in [("good", -1), ("poor", 1)]:
    example_id = max(
        (i for i in labels if labels[i]["label"] == label),
        key=lambda i: direction * (predictions[i]["score"] - labels[i]["score"]),
    )
    print(label, "—", texts[example_id]["text"])
    print(f"Reference: {labels[example_id]['score']:.3f}")
    print(f"Predicted: {predictions[example_id]['score']:.3f}\n")

good — Om du inte hinner i dag kan vi ses på torsdag i stället.
Reference: 0.950
Predicted: 0.459

poor — Vi måste köper mjölk och äggar.
Reference: 0.200
Predicted: 0.718



In [10]:
from google.colab import files

files.download("experiment_augmented/svr.joblib")
files.download("data/train_augmented.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

packages = ["torch", "transformers", "scikit-learn", "numpy", "scipy", "joblib"]
Path("requirements.txt").write_text(
    "\n".join(f"{name}=={version(name)}" for name in packages) + "\n",
    encoding="utf-8",
)

with ZipFile("github_code.zip", "w") as archive:
    for filename in ["train.py", "predict.py", "evaluate.py", "requirements.txt"]:
        archive.write(filename)

files.download("github_code.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>